<a href="https://colab.research.google.com/github/K415mm/mcp_course_workshops/blob/main/Workshop_05_FastMCP_Deploy/05_FastMCP_Deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Workshop 5: FastMCP Deployment

Welcome to the final workshop of the RAISEGUARD Advanced Agentic AI Course!

**Goal:** Go from zero to a fully deployed Cyber Defense Assistant using FastMCP. We will cover the complete development lifecycle: writing safe read-only tools, adding a highly destructive quarantine tool, and wrapping it in an autonomous workflow!

### Setup & Libraries
Run the cell below to install our dependencies.

In [ ]:
!pip install groq langchain-groq mcp requests -q
print("✅ Libraries installed successfully!")

### API Keys
For this final capstone, we only need the LLM!

1. **Groq API Key** (For our LLM Brain)

Add it to the Google Colab **Secrets (🔑)** tab on the left menu, then run this cell.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY successfully loaded!")
except userdata.SecretNotFoundError:
    print("⚠️ WARNING: GROQ_API_KEY missing from Colab Secrets.")
    os.environ["GROQ_API_KEY"] = ""

--- 
## 💾 Lab Data Setup: The Suspicious File
Run this block to write a fake "malicious payload" to the disk, which our agent will investigate and eventually quarantine.

In [ ]:
import os

output_path = "suspicious_payload.exe"
os.makedirs("quarantine_folder", exist_ok=True)

content = b"MZ\x90\x00\x03\x00\x00\x00\x04\x00\x00\x00\xFF\xFF"
content += b"Invoke-Mimikatz \n DownloadCrate \n reverse_shell"

with open(output_path, "wb") as f:
    f.write(content)

print(f"✅ Sample file created: {output_path} ({len(content)} bytes)")

--- 
## 🟢 Beginner Profile: Building Read-Only Analysis Tools
Let's write a few highly reliable tool functions to extract IOCs from alerts and summarize log data.

In [ ]:
import re
import hashlib

def extract_iocs_from_text(text: str) -> dict:
    """Extract all IPv4 addresses, domains, and file hashes from a raw alert text."""
    ipv4 = re.findall(r"\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\b", text)
    domains = re.findall(r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}\b", text)
    hashes = re.findall(r"\b[a-fA-F0-9]{32,64}\b", text)
    
    domains = [d for d in domains if "." in d and not re.match(r"^\d+\.\d+$", d)]
    return {
        "ipv4": list(set(ipv4)),
        "domains": list(set(domains)),
        "hashes": list(set(hashes)),
        "status": "ok"
    }

print(extract_iocs_from_text("Alert: 185.220.101.45 connected to evil.net Hash: a3f9d1..."))

--- 
## 🟡 Intermediate Profile: Exposing Destructive Tools via FastMCP
Now we build our `FastMCP` server. In this server, we will deliberately include a **DESTRUCTIVE** action: `quarantine_file`. We must enforce an approval gate here!

In [ ]:
from mcp.server.fastmcp import FastMCP
import shutil

mcp = FastMCP("Cyber Defense Assistant")

@mcp.tool()
def mcp_extract_iocs(alert_text: str) -> dict:
    """Extract IOCs like IP, Domain, and Hashes from text."""
    return extract_iocs_from_text(alert_text)

@mcp.tool()
def check_file_type(file_path: str) -> dict:
    """Detect the type of a file based on magic bytes."""
    try:
        with open(file_path, "rb") as f:
            header = f.read(16)
        if header.startswith(b"MZ"):
            return {"file": file_path, "detected": "Windows PE", "status": "ok"}
        return {"file": file_path, "detected": "Unknown", "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

@mcp.tool()
def quarantine_file(file_path: str, reason: str, approved_by: str) -> dict:
    """[DESTRUCTIVE] Move a file to the quarantine directory.
    REQUIRES HUMAN APPROVAL — do not call this tool without explicit instruction!
    Provide the reason and the name of the authorizing analyst."""
    try:
        if not approved_by:
            return {"status": "error", "reason": "approved_by is required!"}
        
        filename = os.path.basename(file_path)
        dest = os.path.join("quarantine_folder", f"QUARANTINED_{filename}")
        shutil.move(file_path, dest)
        
        return {
            "status": "quarantined",
            "quarantine_path": dest,
            "reason": reason,
            "approved_by": approved_by
        }
    except Exception as e:
        return {"status": "error", "reason": str(e)}

print("✅ FastMCP Server initialized with tools:", [t.name for t in mcp._tool_manager.get_tools()])

--- 
## 🔴 Advanced Profile: The Cyber Defense Copilot
We are launching our Copilot workflow. Watch how the agent processes the alert, asks for approval, and then executes the destructive action!

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

# Langchain compatible tools
@tool
def ai_extract_iocs(text: str) -> str:
    """Extract IOCs like IP, Domain, and Hashes from text."""
    return str(mcp_extract_iocs(text))

@tool
def ai_check_file(path: str) -> str:
    """Detect the type of a file."""
    return str(check_file_type(path))

@tool
def ai_quarantine(path: str, reason: str, approver: str) -> str:
    """[DESTRUCTIVE] Move a file to the quarantine directory. Requires Approver!"""
    return str(quarantine_file(path, reason, approver))

tools = [ai_extract_iocs, ai_check_file, ai_quarantine]

agent = initialize_agent(
    tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

print("🚀 Copilot Agent Ready!")

In [ ]:
alert_text = """Alert: Outbound connection from 192.168.1.55 to 185.220.101.45.
DNS request for update-secure-patch.net observed.
File hash: 3395856ce81f2b7382dee72602f798b642f14d8 attached."""

prompt = f"""
I need you to triage this alert:
1. Extract IOCs from the raw alert text.
2. Check the file type of suspicious_payload.exe on the local disk.
3. Wait, ACTUALLY I approve quarantine. QUARANTINE suspicious_payload.exe immediately!
   Approver: Kais (Lead Analyst)
   Reason: PE File matching known malicious alert pattern.

Raw alert: {alert_text}
"""

# Kick off the autonomous loop!
agent.run(prompt)